# 📊 Análisis Exploratorio de Datos (EDA)
## Tesis: Asistente Conversacional Inteligente para iTimeControl

**Objetivo:** Caracterizar el corpus documental extraído de los PDFs del sistema iTimeControl,
identificar patrones, distribuciones clave y riesgos potenciales antes del fine-tuning y la indexación RAG.

---
**Estructura del notebook:**
1. Configuración e importaciones
2. Carga del corpus
3. Estadísticas descriptivas básicas
4. Distribución de variables clave
5. Análisis de vocabulario y términos del dominio
6. Análisis del dataset de fine-tuning
7. ⚠️ Riesgos identificados (leakage, desbalance, drift)
8. Conclusiones y recomendaciones


---
## 1. Configuración e Importaciones

In [ ]:
import sys, json, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

# Rutas del proyecto
ROOT         = Path('..')
PROCESSED    = ROOT / 'data' / 'processed'
CHUNKS_DIR   = ROOT / 'data' / 'chunks'
DATASETS_DIR = ROOT / 'data' / 'datasets'

print('✅ Configuración lista')
print(f'   Procesados : {PROCESSED}')
print(f'   Chunks     : {CHUNKS_DIR}')
print(f'   Datasets   : {DATASETS_DIR}')

---
## 2. Carga del Corpus

In [ ]:
# ── Cargar textos procesados ──────────────────────────────────────────────────
txt_files = sorted(PROCESSED.glob('*.txt'))
print(f'Archivos de texto encontrados: {len(txt_files)}')

corpus = []
for f in txt_files:
    text = f.read_text(encoding='utf-8')
    words = text.split()
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 20]
    corpus.append({
        'filename'     : f.name,
        'text'         : text,
        'char_count'   : len(text),
        'word_count'   : len(words),
        'sentence_count': len(sentences),
        'unique_words' : len(set(w.lower() for w in words)),
        'avg_word_len' : np.mean([len(w) for w in words]) if words else 0,
        'avg_sent_len' : np.mean([len(s.split()) for s in sentences]) if sentences else 0,
    })

df_corpus = pd.DataFrame(corpus)
print(df_corpus[['filename','word_count','sentence_count','unique_words']].to_string(index=False))

In [ ]:
# ── Cargar chunks ──────────────────────────────────────────────────────────────
master = CHUNKS_DIR / 'all_chunks.json'
chunks_available = master.exists()

if chunks_available:
    with open(master, encoding='utf-8') as f:
        chunks_raw = json.load(f)
    df_chunks = pd.DataFrame(chunks_raw)
    print(f'Chunks cargados: {len(df_chunks)}')
    print(df_chunks[['source','char_count','word_count']].describe().round(2))
else:
    print('⚠️  Chunks no generados aún — ejecuta: python src/preprocessing/chunker.py')
    df_chunks = pd.DataFrame()

# ── Cargar dataset QA ─────────────────────────────────────────────────────────
train_file = DATASETS_DIR / 'train.jsonl'
dataset_available = train_file.exists()

if dataset_available:
    qa_records = [json.loads(l) for l in train_file.read_text().splitlines() if l.strip()]
    df_qa = pd.DataFrame(qa_records)
    print(f'\nPares QA (train): {len(df_qa)}')
else:
    print('⚠️  Dataset QA no generado — ejecuta: python src/preprocessing/dataset_builder.py')
    df_qa = pd.DataFrame()

---
## 3. Estadísticas Descriptivas Básicas

In [ ]:
print('='*55)
print('  ESTADÍSTICAS DESCRIPTIVAS DEL CORPUS iTimeControl')
print('='*55)

total_words    = df_corpus['word_count'].sum()
total_chars    = df_corpus['char_count'].sum()
total_sents    = df_corpus['sentence_count'].sum()
total_unique   = len(set(' '.join(df_corpus['text']).lower().split()))
lexical_richness = total_unique / total_words if total_words else 0

stats = [
    ('Documentos PDF procesados',      len(df_corpus)),
    ('Total de palabras',              f'{total_words:,}'),
    ('Total de caracteres',            f'{total_chars:,}'),
    ('Total de oraciones',             f'{total_sents:,}'),
    ('Vocabulario único (tokens)',      f'{total_unique:,}'),
    ('Riqueza léxica (TTR)',           f'{lexical_richness:.3f}'),
    ('Promedio palabras/documento',    f'{df_corpus["word_count"].mean():.0f}'),
    ('Promedio oraciones/documento',   f'{df_corpus["sentence_count"].mean():.0f}'),
    ('Longitud promedio de oración',   f'{df_corpus["avg_sent_len"].mean():.1f} palabras'),
]
if chunks_available:
    stats += [
        ('Total de chunks (RAG)',          f'{len(df_chunks):,}'),
        ('Promedio palabras/chunk',        f'{df_chunks["word_count"].mean():.1f}'),
        ('Mediana palabras/chunk',         f'{df_chunks["word_count"].median():.1f}'),
    ]
if dataset_available:
    stats.append(('Pares QA generados (train)',    f'{len(df_qa):,}'))

for label, value in stats:
    print(f'  {label:<40} {value}')
print('='*55)

In [ ]:
# Tabla resumen por documento
display_cols = ['filename', 'word_count', 'sentence_count', 'unique_words', 'avg_sent_len']
rename_map   = {
    'filename'      : 'Documento',
    'word_count'    : 'Palabras',
    'sentence_count': 'Oraciones',
    'unique_words'  : 'Vocab. único',
    'avg_sent_len'  : 'Long. prom. orac.',
}
summary_table = df_corpus[display_cols].rename(columns=rename_map)
summary_table['Long. prom. orac.'] = summary_table['Long. prom. orac.'].round(1)
print('\nResumen por documento:')
print(summary_table.to_string(index=False))

---
## 4. Distribución de Variables Clave

In [ ]:
# ── 4.1 Distribución de palabras por documento ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribución de métricas por documento — Corpus iTimeControl', fontsize=13, y=1.02)

metrics = [
    ('word_count',     'Palabras por documento',   COLORS[0]),
    ('sentence_count', 'Oraciones por documento',  COLORS[1]),
    ('unique_words',   'Vocabulario único',         COLORS[2]),
]

for ax, (col, title, color) in zip(axes, metrics):
    vals = df_corpus[col]
    ax.bar(df_corpus['filename'].apply(lambda x: x[:25]), vals, color=color, edgecolor='white', linewidth=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Documento')
    ax.set_ylabel('Cantidad')
    ax.tick_params(axis='x', rotation=30)
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals)*0.01, f'{v:,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../logs/eda_distribucion_corpus.png', bbox_inches='tight')
plt.show()
print('Gráfica guardada en logs/eda_distribucion_corpus.png')

In [ ]:
# ── 4.2 Distribución de longitud de chunks ────────────────────────────────────
if chunks_available and len(df_chunks) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Distribución de longitud de chunks (RAG)', fontsize=13)

    # Histograma de palabras
    ax = axes[0]
    ax.hist(df_chunks['word_count'], bins=30, color=COLORS[0], edgecolor='white', linewidth=0.6)
    ax.axvline(df_chunks['word_count'].mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Media: {df_chunks["word_count"].mean():.0f}')
    ax.axvline(df_chunks['word_count'].median(), color='orange', linestyle=':',  linewidth=1.5, label=f'Mediana: {df_chunks["word_count"].median():.0f}')
    ax.set_xlabel('Palabras por chunk')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Histograma de palabras/chunk')
    ax.legend()

    # Boxplot por fuente
    ax = axes[1]
    sources = df_chunks['source'].apply(lambda x: Path(x).stem[:20] if x else 'unknown')
    df_chunks['source_label'] = sources
    df_chunks.boxplot(column='word_count', by='source_label', ax=ax, patch_artist=True)
    ax.set_title('Distribución palabras/chunk por fuente')
    ax.set_xlabel('Fuente')
    ax.set_ylabel('Palabras')
    plt.sca(ax)
    plt.xticks(rotation=30)
    plt.suptitle('')

    plt.tight_layout()
    plt.savefig('../logs/eda_distribucion_chunks.png', bbox_inches='tight')
    plt.show()
    print('Gráfica guardada en logs/eda_distribucion_chunks.png')

    # Estadísticas de chunks
    print('\nEstadísticas de chunks:')
    print(df_chunks['word_count'].describe().round(2).to_string())
else:
    print('⚠️  Ejecuta chunker.py para ver esta gráfica.')

In [ ]:
# ── 4.3 Análisis de longitud de instrucciones y respuestas del dataset QA ──
if dataset_available and len(df_qa) > 0:
    df_qa['instruction_len'] = df_qa['instruction'].apply(lambda x: len(str(x).split()))
    df_qa['output_len']      = df_qa['output'].apply(lambda x: len(str(x).split()))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Distribución de longitudes en el Dataset QA (Fine-Tuning)', fontsize=13)

    for ax, (col, label, color) in zip(axes, [
        ('instruction_len', 'Palabras en instrucción (pregunta)', COLORS[3]),
        ('output_len',      'Palabras en output (respuesta)',     COLORS[2]),
    ]):
        ax.hist(df_qa[col], bins=25, color=color, edgecolor='white', linewidth=0.6)
        ax.axvline(df_qa[col].mean(),   color='black',  linestyle='--', linewidth=1.5, label=f'Media: {df_qa[col].mean():.0f}')
        ax.axvline(df_qa[col].median(), color='gray',   linestyle=':',  linewidth=1.5, label=f'Mediana: {df_qa[col].median():.0f}')
        ax.set_xlabel(label)
        ax.set_ylabel('Frecuencia')
        ax.set_title(label)
        ax.legend()

    plt.tight_layout()
    plt.savefig('../logs/eda_distribucion_qa.png', bbox_inches='tight')
    plt.show()
    print('Gráfica guardada en logs/eda_distribucion_qa.png')
else:
    print('⚠️  Ejecuta dataset_builder.py para ver esta gráfica.')

---
## 5. Análisis de Vocabulario y Términos del Dominio

In [ ]:
# ── 5.1 Términos más frecuentes del dominio ───────────────────────────────────
STOPWORDS_ES = {
    'de','la','el','en','y','a','los','del','se','las','por','un','con','una',
    'es','al','lo','para','como','su','más','o','pero','sus','le','ya','o',
    'que','no','este','esta','esta','son','si','fue','ha','hay','ser','tiene',
    'también','cada','cuando','donde','entre','sobre','hasta','desde','todo','
    todos','puede','puede','hacer','así','muy','mismo','sin','otro','otra',
    'dentro','través','número','lugar','primer','segunda','vez','forma'
}

all_text = ' '.join(df_corpus['text'].tolist()).lower()
tokens = re.findall(r'\b[a-záéíóúñü]{4,}\b', all_text)
filtered = [t for t in tokens if t not in STOPWORDS_ES]
freq = Counter(filtered)

# Top 20
top_terms = freq.most_common(20)
terms, counts = zip(*top_terms) if top_terms else ([], [])

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(terms)), counts, color=COLORS[0], edgecolor='white')
ax.set_yticks(range(len(terms)))
ax.set_yticklabels(terms, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Frecuencia')
ax.set_title('Top 20 términos más frecuentes en el corpus iTimeControl', fontsize=12)
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + max(counts)*0.005, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../logs/eda_top_terminos.png', bbox_inches='tight')
plt.show()
print('Gráfica guardada en logs/eda_top_terminos.png')

In [ ]:
# ── 5.2 Cobertura de términos clave del dominio iTimeControl ─────────────────
DOMAIN_TERMS = {
    'Asistencia':   ['asistencia','marcacion','entrada','salida','registro','fichaje'],
    'Empleados':    ['empleado','trabajador','personal','usuario','operario'],
    'Horarios':     ['horario','turno','jornada','calendario','schedule'],
    'Reportes':     ['reporte','informe','estadistica','exportar','resumen'],
    'Permisos':     ['permiso','vacacion','licencia','ausencia','justificacion'],
    'Sistema':      ['sistema','configuracion','modulo','interfaz','parametro'],
    'Calculo':      ['calculo','hora','extra','descuento','tardanza','minuto'],
    'Autenticacion':['clave','contrasena','acceso','login','autenticacion'],
}

coverage = {}
for category, terms in DOMAIN_TERMS.items():
    total = sum(freq.get(t, 0) for t in terms)
    found = sum(1 for t in terms if freq.get(t, 0) > 0)
    coverage[category] = {'frequency': total, 'terms_found': found, 'total_terms': len(terms)}

df_cov = pd.DataFrame(coverage).T
df_cov['coverage_pct'] = (df_cov['terms_found'] / df_cov['total_terms'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cobertura de términos del dominio iTimeControl', fontsize=13)

# Frecuencia por categoría
ax = axes[0]
bars = ax.bar(df_cov.index, df_cov['frequency'], color=COLORS[:len(df_cov)], edgecolor='white')
ax.set_title('Frecuencia total por categoría semántica')
ax.set_ylabel('Frecuencia')
ax.tick_params(axis='x', rotation=40)
for bar, v in zip(bars, df_cov['frequency']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(df_cov['frequency'])*0.01,
            f'{v:,}', ha='center', fontsize=8)

# % de cobertura
ax = axes[1]
colors_cov = ['#55A868' if p >= 80 else '#DD8452' if p >= 50 else '#C44E52'
              for p in df_cov['coverage_pct']]
bars = ax.bar(df_cov.index, df_cov['coverage_pct'], color=colors_cov, edgecolor='white')
ax.axhline(80, color='green',  linestyle='--', linewidth=1, label='80% objetivo')
ax.axhline(50, color='orange', linestyle='--', linewidth=1, label='50% mínimo')
ax.set_title('% de términos clave encontrados por categoría')
ax.set_ylabel('Cobertura (%)')
ax.set_ylim(0, 110)
ax.tick_params(axis='x', rotation=40)
ax.legend(fontsize=9)
for bar, v in zip(bars, df_cov['coverage_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{v}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../logs/eda_cobertura_dominio.png', bbox_inches='tight')
plt.show()

print('\nCobertura por categoría:')
print(df_cov.to_string())

---
## 6. Análisis del Dataset de Fine-Tuning

In [ ]:
if dataset_available and len(df_qa) > 0:
    # ── Splits train/val/test ────────────────────────────────────────────────
    split_counts = {}
    for split in ['train', 'val', 'test']:
        f = DATASETS_DIR / f'{split}.jsonl'
        if f.exists():
            n = len([l for l in f.read_text().splitlines() if l.strip()])
            split_counts[split] = n

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Análisis del Dataset de Fine-Tuning — iTimeControl QA', fontsize=13)

    # Distribución train/val/test
    ax = axes[0]
    labels = list(split_counts.keys())
    sizes  = list(split_counts.values())
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, autopct='%1.1f%%',
        colors=[COLORS[2], COLORS[1], COLORS[3]], startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    ax.set_title('Distribución Train / Val / Test')
    for i, (label, size) in enumerate(zip(labels, sizes)):
        ax.text(0, -1.4 + i*0.18, f'{label}: {size:,} registros',
                ha='center', fontsize=10)

    # Scatter: longitud instrucción vs longitud respuesta
    ax = axes[1]
    sc = ax.scatter(
        df_qa['instruction_len'], df_qa['output_len'],
        alpha=0.4, s=20, c=df_qa['output_len'], cmap='Blues'
    )
    ax.set_xlabel('Palabras en instrucción')
    ax.set_ylabel('Palabras en respuesta')
    ax.set_title('Longitud instrucción vs respuesta')
    plt.colorbar(sc, ax=ax, label='Palabras en respuesta')

    plt.tight_layout()
    plt.savefig('../logs/eda_dataset_qa.png', bbox_inches='tight')
    plt.show()

    print(f'\nEstadísticas del dataset QA:')
    print(df_qa[['instruction_len','output_len']].describe().round(2).to_string())
else:
    print('⚠️  Ejecuta dataset_builder.py para ver esta sección.')

---
## 7. ⚠️ Riesgos Identificados

Esta sección evalúa los tres riesgos principales en proyectos de NLP/LLM:
**Data Leakage**, **Desbalance de clases**, y **Concept Drift**.


In [ ]:
print('='*60)
print('  ANÁLISIS DE RIESGOS — EDA iTimeControl')
print('='*60)

risks = []

# ── RIESGO 1: Data Leakage ────────────────────────────────────────────────────
print('\n🔴 RIESGO 1: Data Leakage')
print('-'*50)

if dataset_available and len(df_qa) > 0:
    # Verificar si alguna instrucción aparece igual en train y test
    train_instructions = set(df_qa['instruction'].str.lower().str.strip())

    test_file = DATASETS_DIR / 'test.jsonl'
    leakage_count = 0
    if test_file.exists():
        test_records = [json.loads(l) for l in test_file.read_text().splitlines() if l.strip()]
        test_instructions = [r['instruction'].lower().strip() for r in test_records]
        leakage_count = sum(1 for t in test_instructions if t in train_instructions)
        leakage_pct = leakage_count / len(test_instructions) * 100 if test_instructions else 0
        print(f'  Instrucciones en test que existen en train: {leakage_count} ({leakage_pct:.1f}%)')
        if leakage_pct > 5:
            print('  ⚠️  RIESGO ALTO: más del 5% de leakage detectado.')
            risks.append(('Data Leakage', 'ALTO', f'{leakage_pct:.1f}% solapamiento train-test'))
        else:
            print('  ✅ Leakage dentro de límites aceptables (< 5%).')
            risks.append(('Data Leakage', 'BAJO', f'{leakage_pct:.1f}% solapamiento'))

    # Verificar que chunks y QA provienen de la misma fuente (esperado)
    print('  Fuente de datos: PDFs procesados del sistema iTimeControl')
    print('  Las preguntas se generan desde el mismo texto → riesgo de overlap semántico')
    print('  Mitigación: usar preguntas diversas en benchmark; evaluar con métricas de generalización')
else:
    print('  No disponible — genera el dataset primero')
    risks.append(('Data Leakage', 'NO EVALUADO', 'Dataset no disponible'))

In [ ]:
# ── RIESGO 2: Desbalance ──────────────────────────────────────────────────────
print('\n🟡 RIESGO 2: Desbalance de datos')
print('-'*50)

if chunks_available and len(df_chunks) > 0:
    # Distribución de chunks por fuente
    source_dist = df_chunks['source'].value_counts()
    max_src  = source_dist.max()
    min_src  = source_dist.min()
    ratio    = max_src / min_src if min_src > 0 else float('inf')

    print(f'  Chunks por fuente:')
    for src, cnt in source_dist.items():
        pct = cnt / len(df_chunks) * 100
        bar = '█' * int(pct / 2)
        print(f'    {Path(src).stem[:30]:<32} {cnt:4d} ({pct:5.1f}%) {bar}')

    print(f'\n  Ratio máx/mín: {ratio:.1f}x')
    if ratio > 5:
        print('  ⚠️  RIESGO ALTO: Un documento domina el corpus (ratio > 5x).')
        print('  Mitigación: Aplicar sub-muestreo de la fuente dominante o augmentation de la minoritaria.')
        risks.append(('Desbalance', 'ALTO', f'Ratio {ratio:.1f}x entre fuentes'))
    elif ratio > 2:
        print('  ⚠️  RIESGO MEDIO: Leve desbalance entre fuentes (ratio 2–5x).')
        print('  Mitigación: Monitorear que el modelo no privilegie una sola fuente.')
        risks.append(('Desbalance', 'MEDIO', f'Ratio {ratio:.1f}x entre fuentes'))
    else:
        print('  ✅ Distribución balanceada entre fuentes (ratio < 2x).')
        risks.append(('Desbalance', 'BAJO', f'Ratio {ratio:.1f}x'))

    # Distribución de longitudes (detectar chunks outliers)
    q1  = df_chunks['word_count'].quantile(0.25)
    q3  = df_chunks['word_count'].quantile(0.75)
    iqr = q3 - q1
    outliers = df_chunks[(df_chunks['word_count'] < q1 - 1.5*iqr) |
                          (df_chunks['word_count'] > q3 + 1.5*iqr)]
    pct_out = len(outliers) / len(df_chunks) * 100
    print(f'\n  Chunks con longitud outlier (IQR): {len(outliers)} ({pct_out:.1f}%)')
    if pct_out > 10:
        print('  ⚠️  RIESGO MEDIO: Muchos chunks con tamaño anómalo.')
        print('  Mitigación: Ajustar chunk_size y chunk_overlap en config.yaml.')
else:
    print('  No disponible — ejecuta chunker.py primero')
    risks.append(('Desbalance', 'NO EVALUADO', 'Chunks no disponibles'))

In [ ]:
# ── RIESGO 3: Concept Drift ───────────────────────────────────────────────────
print('\n🔵 RIESGO 3: Concept Drift')
print('-'*50)

# Para un sistema empresarial como iTimeControl el drift puede ocurrir cuando:
# a) Se actualiza el software y cambia la terminología
# b) La empresa cambia sus procesos internos

drift_indicators = [
    ('Múltiples versiones de documentos', len(txt_files) > 1,
     'Si hay documentos de distintas versiones, puede haber terminología inconsistente'),
    ('Fechas de documentos diferentes', False,  # requeriría metadata de PDF
     'Documentos creados en distintas épocas pueden tener vocabulario diferente'),
    ('Vocabulario inconsistente', False,
     'Ej: el sistema usa "marcación" en un doc y "fichaje" en otro'),
]

# Detectar inconsistencias de vocabulario entre documentos
if len(txt_files) > 1:
    doc_vocabs = []
    for row in df_corpus.itertuples():
        tokens_doc = set(re.findall(r'\b[a-záéíóúñü]{4,}\b', row.text.lower()))
        tokens_doc -= STOPWORDS_ES
        doc_vocabs.append(tokens_doc)

    if len(doc_vocabs) >= 2:
        intersection = doc_vocabs[0].intersection(*doc_vocabs[1:])
        union        = doc_vocabs[0].union(*doc_vocabs[1:])
        jaccard      = len(intersection) / len(union) if union else 1.0
        print(f'  Similitud Jaccard entre vocabularios de documentos: {jaccard:.3f}')
        if jaccard < 0.3:
            print('  ⚠️  RIESGO ALTO: Vocabularios muy distintos entre documentos (Jaccard < 0.3)')
            risks.append(('Concept Drift', 'ALTO', f'Jaccard vocabulario: {jaccard:.3f}'))
        elif jaccard < 0.6:
            print('  ⚠️  RIESGO MEDIO: Vocabulario parcialmente inconsistente')
            risks.append(('Concept Drift', 'MEDIO', f'Jaccard vocabulario: {jaccard:.3f}'))
        else:
            print('  ✅ Vocabulario consistente entre documentos (Jaccard ≥ 0.6)')
            risks.append(('Concept Drift', 'BAJO', f'Jaccard vocabulario: {jaccard:.3f}'))

print('\n  Mitigaciones recomendadas:')
print('  - Versionar los documentos fuente junto con el modelo')
print('  - Re-indexar la base vectorial cuando se actualice iTimeControl')
print('  - Evaluar periódicamente con nuevas preguntas operativas')
risks.append(('Drift futuro', 'LATENTE', 'Actualización de versiones del sistema'))

In [ ]:
# ── Tabla resumen de riesgos ──────────────────────────────────────────────────
print('\n' + '='*60)
print('  RESUMEN DE RIESGOS IDENTIFICADOS')
print('='*60)

COLOR_MAP = {'ALTO': '🔴', 'MEDIO': '🟡', 'BAJO': '🟢',
             'LATENTE': '🔵', 'NO EVALUADO': '⚪'}

df_risks = pd.DataFrame(risks, columns=['Riesgo', 'Nivel', 'Descripción'])
df_risks['Icono'] = df_risks['Nivel'].map(lambda x: COLOR_MAP.get(x, '⚪'))

for _, row in df_risks.iterrows():
    print(f'  {row["Icono"]} {row["Nivel"]:<15} | {row["Riesgo"]:<20} | {row["Descripción"]}')

# Visualización
level_order = ['ALTO', 'MEDIO', 'BAJO', 'LATENTE', 'NO EVALUADO']
color_map_viz = {'ALTO': '#C44E52', 'MEDIO': '#DD8452', 'BAJO': '#55A868',
                 'LATENTE': '#4C72B0', 'NO EVALUADO': '#cccccc'}

fig, ax = plt.subplots(figsize=(10, 4))
for i, (_, row) in enumerate(df_risks.iterrows()):
    color = color_map_viz.get(row['Nivel'], '#cccccc')
    ax.barh(i, 1, color=color, edgecolor='white', height=0.6)
    ax.text(0.02, i, f"{row['Riesgo']} — {row['Nivel']}: {row['Descripción']}",
            va='center', fontsize=9.5, color='white', fontweight='bold')

ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(0, 1)
ax.set_title('Mapa de Riesgos — EDA iTimeControl Assistant', fontsize=12, pad=12)

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in color_map_viz.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../logs/eda_mapa_riesgos.png', bbox_inches='tight')
plt.show()
print('\nGráfica guardada en logs/eda_mapa_riesgos.png')

---
## 8. Conclusiones y Recomendaciones

In [ ]:
print('='*60)
print('  CONCLUSIONES DEL EDA')
print('='*60)
print(f'''
CORPUS:
  • Se procesaron {len(df_corpus)} documento(s) del sistema iTimeControl.
  • El corpus total contiene {total_words:,} palabras y {total_unique:,} tokens únicos.
  • La riqueza léxica (TTR) de {lexical_richness:.3f} indica un vocabulario
    {'especializado y repetitivo (propio de manuales técnicos)' if lexical_richness < 0.1 else 'relativamente diverso'}.

CHUNKS (RAG):
  {'• Se generaron ' + str(len(df_chunks)) + ' chunks listos para indexación.' if chunks_available else '• Pendiente: ejecutar chunker.py'}

DATASET (Fine-Tuning):
  {'• Se generaron ' + str(len(df_qa)) + ' pares QA para entrenamiento.' if dataset_available else '• Pendiente: ejecutar dataset_builder.py'}

RIESGOS:
  • Revisar el mapa de riesgos generado en logs/eda_mapa_riesgos.png

RECOMENDACIONES:
  1. Añadir más documentos de iTimeControl si están disponibles (manuales
     de módulos específicos, FAQs, guías de administrador).
  2. Revisar manualmente ~20 pares QA para validar calidad antes del fine-tuning.
  3. Ajustar chunk_size en config.yaml según la distribución observada.
  4. Documentar la versión del software iTimeControl que corresponde a los PDFs.
  5. Crear un conjunto de preguntas de evaluación escritas por el tutor o usuario real.
''')